In [7]:
import os
import subprocess
import time
import sys
import threading
import socket
from urllib.request import Request, urlopen
import re
# Function to install packages
def install_packages():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit", "pyngrok", "pyjwt", "watchdog"])
print("Installing required packages...")
install_packages()
# Import after installation
from pyngrok import ngrok
# --- Create Streamlit Config for Dark Theme ---
os.makedirs(".streamlit", exist_ok=True)
config_toml = """
[theme]
base="dark"
primaryColor="#4F8BF9"
backgroundColor="#0E1117"
secondaryBackgroundColor="#262730"
textColor="#FAFAFA"
font="sans serif"
[server]
headless = true
"""
with open(".streamlit/config.toml", "w") as f:
    f.write(config_toml)
print("Applied Dark Theme configuration.")

Installing required packages...
Applied Dark Theme configuration.


In [8]:
# --- Create the Streamlit App File ---
app_code = """\
import streamlit as st
import jwt
import datetime
import time
import re
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- Email Configuration ---
# Replace with your Gmail address and App Password
# Create an App Password at: https://myaccount.google.com/apppasswords
SENDER_EMAIL = "arungoshala1@gmail/com"
SENDER_APP_PASSWORD = "ljlf suol efui bepf"
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587

# --- JWT Configuration ---
SECRET_KEY = "super_secret_key_for_demo"  # In production, use environment variable
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
OTP_EXPIRE_MINUTES = 10

# --- JWT Utils ---
def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.datetime.utcnow() + datetime.timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    return encoded_jwt

def verify_token(token: str):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

# --- Validation Utils ---
def is_valid_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'
    try:
        if re.match(pattern, email):
            return True
    except:
        return False
    return False

def is_valid_password(password):
    if len(password) < 8:
        return False
    if not password.isalnum():
        return False
    return True

# --- OTP Utils ---
def generate_otp():
    """Generate a 6-digit OTP."""
    return str(random.randint(100000, 999999))

def send_otp_email(recipient_email: str, otp: str) -> tuple[bool, str]:
    """Send OTP to the given email. Returns (success, error_message)."""
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = "🔐 Infosys SpringBoard – Password Reset OTP"
        msg["From"] = SENDER_EMAIL
        msg["To"] = recipient_email

        html_body = f"""
        <html>
        <body style="font-family: 'Inter', Arial, sans-serif; background:#0E1117; color:#FAFAFA; padding:30px;">
            <div style="max-width:480px; margin:auto; background:#262730; border-radius:12px; padding:36px; text-align:center;">
                <h2 style="color:#4F8BF9; margin-bottom:8px;"> Infosys SpringBoard Intern</h2>
                <p style="color:#aaa; margin-bottom:24px;">Password Reset Request</p>
                <div style="background:#0E1117; border-radius:8px; padding:24px; margin-bottom:24px;">
                    <p style="color:#FAFAFA; margin:0 0 8px 0; font-size:14px;">Your One-Time Password (OTP):</p>
                    <h1 style="color:#4F8BF9; letter-spacing:12px; margin:0; font-size:42px;">{otp}</h1>
                </div>
                <p style="color:#aaa; font-size:13px;">This OTP is valid for <strong style="color:#FAFAFA;">{OTP_EXPIRE_MINUTES} minutes</strong>.</p>
                <p style="color:#aaa; font-size:13px;">If you did not request this, please ignore this email.</p>
            </div>
        </body>
        </html>
        """
        text_body = f"Your OTP for Infosys SpringBoard password reset is: {otp}\\nIt is valid for {OTP_EXPIRE_MINUTES} minutes."

        msg.attach(MIMEText(text_body, "plain"))
        msg.attach(MIMEText(html_body, "html"))

        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.ehlo()
            server.starttls()
            server.login(SENDER_EMAIL, SENDER_APP_PASSWORD)
            server.sendmail(SENDER_EMAIL, recipient_email, msg.as_string())

        return True, ""
    except smtplib.SMTPAuthenticationError:
        return False, "Email authentication failed. Check SENDER_EMAIL and SENDER_APP_PASSWORD in app.py."
    except Exception as e:
        return False, str(e)

def is_otp_valid(email: str, entered_otp: str) -> bool:
    """Validate the entered OTP against the stored one."""
    store = st.session_state.get("otp_store", {})
    if email not in store:
        return False
    record = store[email]
    if datetime.datetime.utcnow() > record["expires"]:
        return False
    return record["otp"] == entered_otp

# --- Session State Init ---
if "jwt_token" not in st.session_state:
    st.session_state["jwt_token"] = None
if "page" not in st.session_state:
    st.session_state["page"] = "login"
if "users" not in st.session_state:
    st.session_state["users"] = {}
if "usernames" not in st.session_state:
    st.session_state["usernames"] = set()
if "otp_store" not in st.session_state:
    st.session_state["otp_store"] = {}
if "reset_email" not in st.session_state:
    st.session_state["reset_email"] = ""

# --- Page Config & Styling ---
st.set_page_config(page_title="Infosys SpringBoard Intern", page_icon="🤖", layout="wide")

st.markdown("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');

        .stApp {
            background-color: #0E1117;
            font-family: 'Inter', sans-serif;
        }
        h1 {
            text-align: center;
            color: #4F8BF9;
            font-family: 'Inter', sans-serif;
            margin-bottom: 0.5rem;
        }
        h3 {
            text-align: center;
            color: #FAFAFA;
            font-weight: 300;
            margin-top: 0;
            font-size: 1.2rem;
        }
        .stButton > button {
            width: 100%;
            border-radius: 8px;
            height: 3em;
            background-color: #4F8BF9;
            color: white;
            font-weight: bold;
            border: none;
            transition: background-color 0.2s;
        }
        .stButton > button:hover {
            background-color: #3b6ccf;
        }
        div[data-testid="stSidebar"] {
            background-color: #262730;
        }
        .otp-info-box {
            background: linear-gradient(135deg, #1a2340, #262730);
            border: 1px solid #4F8BF9;
            border-radius: 10px;
            padding: 16px;
            margin-bottom: 16px;
            text-align: center;
        }
        .user-msg {
            text-align: right;
            background-color: #262730;
            color: white;
            padding: 10px;
            border-radius: 10px;
            margin: 5px;
            display: inline-block;
            max-width: 80%;
            float: right;
            clear: both;
        }
        .bot-msg {
            text-align: left;
            background-color: #4F8BF9;
            color: white;
            padding: 10px;
            border-radius: 10px;
            margin: 5px;
            display: inline-block;
            max-width: 80%;
            float: left;
            clear: both;
        }
    </style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────
#  VIEWS
# ─────────────────────────────────────────────

def login_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("Infosys SpringBoard Intern")
        st.markdown("<h3>Please sign in to continue</h3>", unsafe_allow_html=True)

        with st.form("login_form"):
            email = st.text_input("Email Address")
            password = st.text_input("Password", type="password")
            submitted = st.form_submit_button("Sign In")

            if submitted:
                if email in st.session_state["users"] and \\
                        st.session_state["users"][email]["password"] == password:
                    username = st.session_state["users"][email]["username"]
                    token = create_access_token({"sub": email, "username": username})
                    st.session_state["jwt_token"] = token
                    st.success("Login successful!")
                    time.sleep(0.5)
                    st.rerun()
                else:
                    st.error("Invalid email or password")

        st.markdown("---")
        c1, c2 = st.columns(2)
        with c1:
            if st.button("Forgot Password?"):
                st.session_state["page"] = "forgot_password"
                st.rerun()
        with c2:
            if st.button("Create an Account"):
                st.session_state["page"] = "signup"
                st.rerun()


def signup_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("Create Account")

        with st.form("signup_form"):
            username = st.text_input("Username (Required)")
            email = st.text_input("Email Address (@domain.com required)")
            password = st.text_input("Password (min 8 chars, alphanumeric)", type="password")
            confirm_password = st.text_input("Confirm Password", type="password")
            submitted = st.form_submit_button("Sign Up")

            if submitted:
                errors = []

                if not username:
                    errors.append("Username is mandatory.")
                elif username in st.session_state["usernames"]:
                    errors.append(f"Username '{username}' is already taken.")

                if not email:
                    errors.append("Email is mandatory.")
                elif not is_valid_email(email):
                    errors.append("Invalid Email format (e.g. user@domain.com).")
                elif email in st.session_state["users"]:
                    errors.append(f"Email '{email}' is already registered.")

                if not password:
                    errors.append("Password is mandatory.")
                elif not is_valid_password(password):
                    errors.append("Password must be at least 8 characters long and alphanumeric.")

                if password != confirm_password:
                    errors.append("Passwords do not match.")

                if errors:
                    for error in errors:
                        st.error(error)
                else:
                    st.session_state["users"][email] = {"password": password, "username": username}
                    st.session_state["usernames"].add(username)
                    token = create_access_token({"sub": email, "username": username})
                    st.session_state["jwt_token"] = token
                    st.success("Account created successfully!")
                    time.sleep(1)
                    st.rerun()

        st.markdown("---")
        if st.button("Back to Login"):
            st.session_state["page"] = "login"
            st.rerun()


# ─────────────────────────────────────────────
#  FORGOT PASSWORD – STEP 1: Enter Email
# ─────────────────────────────────────────────
def forgot_password_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("🔐 Forgot Password")
        st.markdown(
            "<h3>Enter your registered email to receive an OTP</h3>",
            unsafe_allow_html=True,
        )

        with st.form("forgot_form"):
            email = st.text_input("Registered Email Address")
            submitted = st.form_submit_button("Send OTP")

            if submitted:
                if not email or not is_valid_email(email):
                    st.error("Please enter a valid email address.")
                elif email not in st.session_state["users"]:
                    st.error("No account found with this email address.")
                else:
                    otp = generate_otp()
                    expires = datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRE_MINUTES)
                    st.session_state["otp_store"][email] = {"otp": otp, "expires": expires}
                    st.session_state["reset_email"] = email

                    with st.spinner("Sending OTP to your email…"):
                        success, err_msg = send_otp_email(email, otp)

                    if success:
                        st.success(f"✅ OTP sent to **{email}**. Please check your inbox (and spam folder).")
                        time.sleep(1.5)
                        st.session_state["page"] = "verify_otp"
                        st.rerun()
                    else:
                        # Show OTP in UI if email fails (dev/demo convenience)
                        st.warning(
                            f"⚠️ Could not send email: {err_msg}\\n\\n"
                            f"**[Demo mode]** Your OTP is: `{otp}`"
                        )

        st.markdown("---")
        if st.button("← Back to Login"):
            st.session_state["page"] = "login"
            st.rerun()


# ─────────────────────────────────────────────
#  FORGOT PASSWORD – STEP 2: Verify OTP
# ─────────────────────────────────────────────
def verify_otp_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])
    reset_email = st.session_state.get("reset_email", "")

    with col2:
        st.title("📧 Verify OTP")
        st.markdown(
            f"<h3>Enter the 6-digit OTP sent to <span style='color:#4F8BF9'>{reset_email}</span></h3>",
            unsafe_allow_html=True,
        )
        st.markdown(
            f"""<div class="otp-info-box">
                <p style="color:#aaa; margin:0; font-size:13px;">
                    ⏱ OTP is valid for <strong style="color:#FAFAFA;">{OTP_EXPIRE_MINUTES} minutes</strong>
                </p>
            </div>""",
            unsafe_allow_html=True,
        )

        with st.form("otp_form"):
            otp_input = st.text_input("Enter OTP", max_chars=6, placeholder="••••••")
            submitted = st.form_submit_button("Verify OTP")

            if submitted:
                if not otp_input:
                    st.error("Please enter the OTP.")
                elif not reset_email:
                    st.error("Session error. Please start over.")
                    st.session_state["page"] = "forgot_password"
                    st.rerun()
                elif is_otp_valid(reset_email, otp_input.strip()):
                    st.success("✅ OTP verified successfully!")
                    # Clear OTP so it can't be reused
                    st.session_state["otp_store"].pop(reset_email, None)
                    time.sleep(0.8)
                    st.session_state["page"] = "reset_password"
                    st.rerun()
                else:
                    st.error("❌ Invalid or expired OTP. Please try again or request a new one.")

        st.markdown("---")
        c1, c2 = st.columns(2)
        with c1:
            if st.button("← Back"):
                st.session_state["page"] = "forgot_password"
                st.rerun()
        with c2:
            if st.button("🔄 Resend OTP"):
                if reset_email and reset_email in st.session_state["users"]:
                    otp = generate_otp()
                    expires = datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRE_MINUTES)
                    st.session_state["otp_store"][reset_email] = {"otp": otp, "expires": expires}
                    with st.spinner("Resending OTP…"):
                        success, err_msg = send_otp_email(reset_email, otp)
                    if success:
                        st.success("✅ New OTP sent!")
                    else:
                        st.warning(f"⚠️ Could not send email: {err_msg}\\n\\n**[Demo mode]** New OTP: `{otp}`")
                else:
                    st.error("Session error. Please start over.")
                    st.session_state["page"] = "forgot_password"
                    st.rerun()


# ─────────────────────────────────────────────
#  FORGOT PASSWORD – STEP 3: Reset Password
# ─────────────────────────────────────────────
def reset_password_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])
    reset_email = st.session_state.get("reset_email", "")

    with col2:
        st.title("🔑 Reset Password")
        st.markdown("<h3>Create a new password for your account</h3>", unsafe_allow_html=True)

        if not reset_email or reset_email not in st.session_state["users"]:
            st.error("Session error. Please start over.")
            if st.button("← Back to Login"):
                st.session_state["page"] = "login"
                st.rerun()
            return

        with st.form("reset_form"):
            new_password = st.text_input(
                "New Password (min 8 chars, alphanumeric)", type="password"
            )
            confirm_password = st.text_input("Confirm New Password", type="password")
            submitted = st.form_submit_button("Reset Password")

            if submitted:
                errors = []
                if not new_password:
                    errors.append("New password is required.")
                elif not is_valid_password(new_password):
                    errors.append(
                        "Password must be at least 8 characters long and contain only alphanumeric characters."
                    )
                if new_password != confirm_password:
                    errors.append("Passwords do not match.")

                if errors:
                    for e in errors:
                        st.error(e)
                else:
                    # Update the password
                    st.session_state["users"][reset_email]["password"] = new_password
                    st.session_state["reset_email"] = ""
                    st.success("✅ Password reset successfully! Please log in with your new password.")
                    time.sleep(1.5)
                    st.session_state["page"] = "login"
                    st.rerun()

        st.markdown("---")
        if st.button("← Back"):
            st.session_state["page"] = "login"
            st.rerun()


# ─────────────────────────────────────────────
#  DASHBOARD
# ─────────────────────────────────────────────
def dashboard_page():
    token = st.session_state.get("jwt_token")
    payload = verify_token(token)

    if not payload:
        st.session_state["jwt_token"] = None
        st.warning("Session expired or invalid. Please login again.")
        time.sleep(1)
        st.rerun()
        return

    username = payload.get("username", "User")

    with st.sidebar:
        st.title("🤖 LLM")
        st.markdown("---")
        if st.button("➕ New Chat", use_container_width=True):
            st.info("Started new chat!")

        st.markdown("### History")
        st.markdown("- Project analysis")
        st.markdown("- NLP")
        st.markdown("---")
        st.markdown("### Settings")
        if st.button("Logout", use_container_width=True):
            st.session_state["jwt_token"] = None
            st.rerun()

    st.title(f"Welcome, {username}!")
    st.markdown("### How can I help you today?")

    chat_placeholder = st.empty()
    with chat_placeholder.container():
        st.markdown(
            '<div class="bot-msg">Hello! I am LLM. Ask me anything about LLM!</div>',
            unsafe_allow_html=True,
        )

    with st.form(key="chat_form", clear_on_submit=True):
        col1, col2 = st.columns([6, 1])
        with col1:
            user_input = st.text_input(
                "Message LLM...",
                placeholder="Ask me anything about LLM...",
                label_visibility="collapsed",
            )
        with col2:
            submit_button = st.form_submit_button("Send")

        if submit_button and user_input:
            st.markdown(
                f'<div class="user-msg">{user_input}</div>', unsafe_allow_html=True
            )
            st.markdown(
                '<div class="bot-msg">I am a demo bot. I received your message!</div>',
                unsafe_allow_html=True,
            )


# ─────────────────────────────────────────────
#  ROUTING
# ─────────────────────────────────────────────
token = st.session_state.get("jwt_token")
if token:
    if verify_token(token):
        dashboard_page()
    else:
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()
else:
    page = st.session_state["page"]
    if page == "signup":
        signup_page()
    elif page == "forgot_password":
        forgot_password_page()
    elif page == "verify_otp":
        verify_otp_page()
    elif page == "reset_password":
        reset_password_page()
    else:
        login_page()
"""
with open("app.py", "w") as f:
    f.write(app_code)
print("Streamlit app code written to 'app.py'")

IndentationError: unindent does not match any outer indentation level (<string>, line 135)

In [ ]:
# --- Wait for Streamlit to Start ---
def wait_for_streamlit(port=8501, timeout=30):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex(('localhost', port))
            if result == 0:
                sock.close()
                return True
            sock.close()
        except Exception:
            pass
        time.sleep(1)
    return False
# --- Ngrok Setup ---
print("\nTo access the app, you need an Ngrok Authtoken.")
print("Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
authtoken = input("Enter your Ngrok Authtoken: ").strip()
if authtoken:
    ngrok.set_auth_token(authtoken)

    # Kill any existing ngrok process
    os.system("pkill ngrok")
    os.system("pkill streamlit")

    # Run Streamlit in the background FIRST
    print("Starting Streamlit...")
    # Using Subprocess.Popen to run in background
    # Redirecting output to /dev/null to keep cell clean
    process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "localhost"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Wait for it to be ready
    if wait_for_streamlit():
        print("Streamlit is active! Connecting Ngrok...")
        # Open a tunnel to the streamlit port 8501
        try:
            public_url = ngrok.connect(8501).public_url
            print(f"\n🚀 Streamlit App is running!")
            print(f"👉 Public URL: {public_url}")
            print("\n(Click the URL above to open the app)")

            # Keep main thread alive
            try:
                # Keep checking if process is alive
                while process.poll() is None:
                    time.sleep(1)
            except KeyboardInterrupt:
                print("Stopping...")
                ngrok.disconnect(public_url)
                process.terminate()
        except Exception as e:
            print(f"Ngrok connection failed: {e}")
            process.terminate()
    else:
        print("Error: Streamlit failed to start in time.")
        process.terminate()
else:
    print("Ngrok Authtoken is required to expose the app publicly.")


To access the app, you need an Ngrok Authtoken.
Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
Enter your Ngrok Authtoken: 39VeSDNVC7IJrtt4WpBEcxepb5m_7e22DsZn38MdLaWVVuYSx
Starting Streamlit...
Streamlit is active! Connecting Ngrok...

🚀 Streamlit App is running!
👉 Public URL: https://electrometrically-syntonous-jeanetta.ngrok-free.dev

(Click the URL above to open the app)


Stopping...
Ngrok connection failed: Remote end closed connection without response
